In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as datetime
import matplotlib.dates as mdates

# Choose some nice levels
plt.rcParams["figure.figsize"] = [16, 9]
plt.rcParams["figure.autolayout"] = True

In [ ]:
commits = pd.read_csv('../data/zephyr_commit_times.csv',names=["sha1", "author_date", "author_email", "committer_date", "committer_email"], header=0)

In [ ]:
commits.author_date= pd.to_datetime(commits.author_date, utc=True)
commits.committer_date = pd.to_datetime(commits.committer_date, utc=True)
commits.committer_date.min()

In [ ]:
resampled=commits.resample("M", on="author_date").count()
resampled

In [ ]:
fig, ax = plt.subplots(figsize=(16,9), constrained_layout=True)
ax.bar(resampled.index, resampled['sha1'], width=35)
            
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(),rotation=45)
             
fig.savefig('../figures/zephyr-commits-per-month.svg', format='svg')     

In [ ]:
activity_max = commits.filter(['committer_date'], axis=1).groupby(commits['author_email']).max().rename(columns={"committer_date":"max_author_date"})
activity_min = commits.filter(['committer_date'], axis=1).groupby(commits['author_email']).min().rename(columns={"committer_date":"min_author_date"})
activity_commits = commits.filter(['committer_date'], axis=1).groupby(commits['author_email']).count().rename(columns={'committer_date':'#commits'})
author_activity = pd.concat([activity_commits,activity_min, activity_max], axis=1, join='inner')
author_activity['span'] = author_activity['max_author_date'] - author_activity['min_author_date']
author_activity['span'].describe()

In [ ]:
# Top 50 longest active authors
veterans=author_activity.sort_values(by='#commits', ascending=False)
veterans.head(50)


In [ ]:
#Onetime Committer
veterans[veterans['span']==pd.Timedelta(0)].count()

In [ ]:
author_activity.sort_values(by='span', ascending=False)['span'].dt.days.to_numpy()

In [ ]:
commits.filter(['committer_date'], axis=1).groupby(commits['author_email']).count().rename(columns={'committer_date':'#commits'})